In [2]:

import pandas as pd
import numpy as np
from scipy.spatial.distance import mahalanobis
from scipy.spatial.distance import euclidean

test_in = pd.read_csv(r"data/test_in - Copy.csv",header=None)
test_out = pd.read_csv(r"data/test_out - Copy.csv",header=None)
test_out = test_out.rename(columns={0:"label"})
train_in = pd.read_csv(r"data/train_in - Copy.csv",header=None)
train_out = pd.read_csv(r"data/train_out - Copy.csv",header=None)
train_out = train_out.rename(columns={0:"label"})
train_set = pd.concat([train_in,train_out],axis=1)
test_set = pd.concat([test_in,test_out],axis=1)
inverse_of_cov = np.linalg.inv(train_in.cov())

In [3]:
centroids = train_set.groupby("label").mean()
results = []

for _, row in test_set.iterrows():
    predicted_mahalanobis_label = None
    predicted_euclidean_label = None
    smallest_mahalanobis_distance = float("inf")
    smallest_euclidean_distance = float("inf")

    for _, cent in centroids.iterrows():
        dist_mahalanobis = mahalanobis(row.iloc[:-1], cent, inverse_of_cov)
        dist_euclidean = euclidean(row.iloc[:-1], cent)
        label = cent.name

        if dist_mahalanobis < smallest_mahalanobis_distance:
            smallest_mahalanobis_distance = dist_mahalanobis
            predicted_mahalanobis_label = label
        if dist_euclidean < smallest_euclidean_distance:
            smallest_euclidean_distance = dist_euclidean
            predicted_euclidean_label = label

    results.append({"prediction_mahalanobis": predicted_mahalanobis_label, "distance_mahalanobis": smallest_mahalanobis_distance,  "prediction_euclidean": predicted_euclidean_label, "distance_euclidean": smallest_euclidean_distance})

prediction_and_distances = pd.DataFrame(results)
test_set = pd.concat([test_set,prediction_and_distances],axis=1)

total_count_test = len(test_set)

accuracy_euclidean = (test_set["label"] == test_set["prediction_euclidean"]).mean()
accuracy_mahalanobis = (test_set["label"] == test_set["prediction_mahalanobis"]).mean()



44821.66049608618
